# 00 · Start here

**raglab**
Ten notebooks that build a retrieval system and the evaluation that judges it.

---

This notebook and the nine that follow it build a retrieval system you can
run, break, measure and defend. Everything executes offline in a few seconds. There is no
dataset to download, no API key to set, and no service to start — the entire retrieval stack
lives inside a `sqlite3.connect(":memory:")` database that disappears when you shut the
kernel down.

### How to run

Press **Run All**. That is the whole procedure. The first cell installs anything missing
(`numpy`, `matplotlib`, `pandas`) and pins every source of randomness so that two runs of the
same notebook produce the same numbers — which matters more than it sounds, because half of
this curriculum is about telling a real improvement from run-to-run noise, and a harness that
is itself unstable cannot teach that.

### What you get

| Notebook | Deck section | What you build and measure |
|---|---|---|
| **00** Start here | — | The toolkit, the corpus, one query end to end |
| **01** Retrieval & evaluation foundations | §1 | The four-stage pipeline; the fault-isolation tree, executed |
| **02** MultiHop-RAG use case | §2 | Question types, evidence lists, manufacturing an eval set |
| **03** RAG system design | §3 | Chunking, index freshness, blue/green, permission-aware retrieval |
| **04** Retrieval methods & reranking | §4 | BM25 from scratch, ANN recall curves, fusion, rerankers |
| **05** LLM context design | §5 | Token budgets, packing, provenance, position effects |
| **06** Evaluation approaches | §6 | Layered metrics, judge calibration, the release gate |
| **07** Cost & token optimisation | §7 | Four token categories, prompt caching, unit economics |
| **08** Agentic search | §8 | Decompose → tool → retrieve → sufficiency → stop; trace scoring |
| **09** Capstone build | Build brief | The brief's five steps, scored against the rubric |

Each notebook follows the same rhythm: **a flowchart of what the code is about to do**, then
the code, then **a summary — a diagram, a decision tree, and that tree read back as a table**
so the rule survives outside the notebook. Sections end with the deck's failure signatures
reproduced on real queries, and the interview questions the panel actually asks.


In [ ]:
# ── One-click setup ──────────────────────────────────────────────────────────
# Installs anything missing, pins the seed, and puts the toolkit on the path.
import pathlib, sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
env = bootstrap()

The optional rows above are expected to be absent. `boto3`, `sentence-transformers` and
`anthropic` unlock live paths — a Bedrock Knowledge Base, a real neural encoder, a model
judge — and every one of them is genuinely optional. Nothing in these notebooks fails
without them.


In [ ]:
# ── AWS Bedrock preflight ────────────────────────────────────────────────────
# Read-only: reports what is configured, makes no AWS calls, bills nothing.
from raglab.bedrock import BedrockConfig, preflight

BEDROCK = preflight(BedrockConfig.from_env())

## The toolkit

Nine modules. The split matters: the *concepts* are implemented in the notebooks where you
can read and change them, and the *infrastructure* lives in the package so the notebooks stay
about retrieval rather than about plumbing.


In [ ]:
from raglab import viz, tables

viz.reset_figures("0.")
tables.reset_tables("0.")

viz.hld(
    [
        dict(name="Data", tone="index", chain=False, nodes=[
            ("corpus.py", "fact graph → documents, chunks, and an eval set whose gold labels "
                          "are true by construction"),
            ("chunking.py", "the deck's seven strategies, with stable chunk ids"),
            ("embed.py", "LSA by default; sentence-transformers or Bedrock Titan when present"),
        ]),
        dict(name="Retrieval", tone="query", chain=False, nodes=[
            ("store.py", "in-memory SQLite: FTS5 lexical index, vector table, NSW graph, "
                         "versioned indexes with an alias"),
            ("retrieve.py", "hybrid fusion, learned reranker, packing under a token cap"),
            ("context.py", "prompt assembly in volatility order, with provenance"),
            ("generate.py", "extractive offline reader; Bedrock or Claude when configured"),
        ]),
        dict(name="Measurement", tone="control", chain=False, nodes=[
            ("metrics.py", "evidence recall, full-chain recall, nDCG, abstention, κ, "
                           "paired bootstrap"),
            ("judge.py", "rubrics as release artefacts, calibration, bias probes"),
            ("trace.py", "queryable trace store; diff two runs of one query"),
            ("pipeline.py", "one config object; evaluate() runs a set and returns rows"),
        ]),
        dict(name="Reference", tone="store", chain=False, nodes=[
            ("catalog.py", "the deck's decision trees and matrices, as executable data"),
            ("trees.py", "a tree renders, tabulates, and runs from one definition"),
            ("viz.py / tables.py", "the diagram and table language used throughout"),
            ("bedrock.py", "Knowledge Base retriever + the local→AWS mapping"),
            ("costs.py", "token categories, prompt caching, latency and unit economics"),
        ]),
    ],
    title="What is in the box",
    kicker="Toolkit map",
    caption="Concepts live in the notebooks; infrastructure lives in the package. "
            "Everything in the Retrieval lane has a documented swap to a managed service.",
)

## Build the whole system in one call

`quickstart()` does what notebooks 01 through 04 do slowly and explicitly: builds the corpus,
chunks it, fits the encoder, writes both indexes, and returns a configured pipeline.
`TUNED` is the configuration notebook 04 arrives at *by measurement* — weighted fusion with
the dense leg at 0.3, a learned cross-encoder over the top 50, k=8 under a 6,000-token
evidence cap. It is not a default anyone should inherit without re-deriving it on their own
corpus, and notebook 04 derives it in front of you.


In [ ]:
import raglab

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)

In [ ]:
from raglab import tables

stats = bundle.stats()
tables.keyvalue(
    [("Documents", f"{stats['documents']:,}"),
     ("Words", f"{stats['words']:,}"),
     ("Chunks (structural)", f"{len(pipe.chunks):,}"),
     ("Sources", ", ".join(f"{k} {v}" for k, v in stats["sources"].items())),
     ("Restricted documents", f"{stats['restricted_docs']} (legal / finance / support only)"),
     ("Eval questions", f"{stats['questions']}"),
     ("Question types", ", ".join(f"{k} {v}" for k, v in stats["question_types"].items())),
     ("Evidence hops", ", ".join(f"{k}-hop: {v}" for k, v in sorted(stats["hops"].items()))),
     ("Frozen slice", f"{sum(1 for q in bundle.questions if q.slice == 'frozen')} questions "
                      f"no tuning run may look at")],
    title="The corpus you will spend the day on",
    kicker="Dataset",
    caption="Generated from a fact graph so every gold evidence label is exact rather than "
            "annotated. The failure modes in the deck are built in on purpose.",
)

### Why a synthetic corpus rather than the real MultiHop-RAG

Three reasons, and they are the same reasons you build a domain eval set instead of
reaching for a public benchmark.

1. **One click.** A notebook that needs a 200 MB download is not a notebook a cohort can run
   in a room with hotel wifi.
2. **Gold labels that are true by construction.** Every question is generated from the fact
   graph, so the evidence list is exact. There is no annotation-error floor underneath the
   numbers you are about to read.
3. **Deliberate failure modes.** A public benchmark contains whatever failures it happens to
   contain. This corpus was built so that every signature in the deck actually fires — the
   lexical gap, the identifier miss, the missing hop, distractor dominance, temporal
   ordering, null questions, and ACL leakage.

The record schema mirrors MultiHop-RAG's exactly, so everything transfers. If you have the
real files, `corpus.load_multihop_rag("corpus.json", "MultiHopRAG.json")` swaps them in and
nothing downstream changes.


In [ ]:
import json
from raglab import tables

q = next(q for q in bundle.questions if q.hops == 2 and q.question_type == "inference")
doc = bundle.by_id(q.evidence_doc_ids[0])

print("A DOCUMENT ─────────────────────────────────────────────────────────────")
print(f"doc_id: {doc.doc_id}   source: {doc.source}   published: {doc.published}")
print(f"acl: {doc.acl}   content_hash: {doc.content_hash}")
print(doc.body[:520], "…\n")

print("AN EVAL RECORD (MultiHop-RAG schema) ───────────────────────────────────")
print(json.dumps(q.as_record(), indent=2)[:700])
print(f"\nhops={q.hops}  difficulty={q.difficulty}  slice={q.slice}  persona={q.persona}")
print(f"note: {q.note}")

The `evidence_list` is the contract. Without it you can only score answers, and answer-only
scoring hides half of your failures: an answer can be right by accident while retrieval
missed every gold document. That single field is what makes Evidence Recall and full-chain
recall possible, and those are the two numbers that predict whether a multi-hop system works.

## One query, end to end


In [ ]:
trace = pipe.run(q.query, qid=q.qid)

print(f"QUESTION   {trace.query}\n")
print(f"STAGES     " + "  ".join(f"{k}={v:.1f}ms" for k, v in trace.stage_ms.items()))
print(f"           {len(trace.candidates)} candidates → {len(trace.packed)} packed "
      f"({sum(b['tokens'] for b in trace.packed)} evidence tokens)\n")
print("PACKED CONTEXT")
for b in trace.packed:
    row = index.get(b["chunk_id"])
    print(f"  [{b['sid']:<3}] {b['score']:.3f}  {b['doc_id']:<10} {row['title'][:52]}")
print(f"\nANSWER     {trace.answer[:300]}")
print(f"CITATIONS  {trace.citations}  →  "
      f"{[trace._packed_obj.resolve(c) for c in trace.citations]}")
print(f"GOLD       {q.answer}")

Every number in that output is stored. The trace carries the candidates and their scores, the
packed context and its token counts, the answer, the citations and the per-stage latency —
which is the deck's requirement that a bad answer can be *replayed*, not merely regretted.
Notebook 03 uses the trace store to diff two runs of the same query; notebook 06 turns
production failures with a human verdict into new regression cases.

## Being straight with you about what is real

A curriculum that quietly simulates its results teaches nothing you can defend in a design
review. Here is the honest inventory.


In [ ]:
import pandas as pd
from raglab import tables

tables.show(pd.DataFrame([
    ["Lexical retrieval", "Real",
     "SQLite FTS5 — a genuine inverted index with SQLite's own BM25 implementation",
     "—"],
    ["Vector search (exact)", "Real",
     "Brute-force cosine over stored float32 vectors; recall = 1.0 by construction", "—"],
    ["Approximate search", "Real",
     "A navigable small-world graph with greedy best-first search; efSearch genuinely trades "
     "recall for visits", "Swap for HNSW / IVF-PQ / a managed index"],
    ["Embeddings", "Real, but weak",
     "Latent semantic analysis: TF-IDF then truncated SVD. Genuine dense retrieval, and "
     "roughly fifty years behind a modern encoder", "SentenceTransformers or Bedrock Titan"],
    ["Reranker", "Real, but small",
     "Logistic regression over eight query–passage pair features, fitted on the dev slice. "
     "Same cost model as a cross-encoder, a fraction of the capacity",
     "cross-encoder/ms-marco, or Bedrock rerank"],
    ["Late interaction", "Real", "MaxSim over token vectors in the encoder's term space", "ColBERT"],
    ["Generation", "Real, but extractive",
     "Selects and cites supporting sentences. Faithful by construction — which means it "
     "cannot hallucinate, and cannot derive a comparison either",
     "Bedrock Converse or the Claude API"],
    ["Hallucination", "Fault injection",
     "UngroundedGenerator exists only to give the judge something real to catch. It is a "
     "fixture, not a claim about how models behave", "—"],
    ["LLM judge", "Real, heuristic",
     "Span-level support checking, deterministic and free. Weaker on nuance than a model "
     "judge, stronger on consistency", "BedrockJudge with the same rubrics"],
    ["Costs and latency", "Modelled",
     "Arithmetic with the deck's illustrative rates, clearly labelled. Provider prices change; "
     "the arithmetic is the transferable part", "Real usage counters from either provider"],
], columns=["Component", "Status", "What it actually is", "Production swap"]),
    title="What is real, what is a stand-in, and what you would swap",
    kicker="Honesty inventory",
    caption="Read the third column before you quote any number from these notebooks outside "
            "the room. The measurement discipline transfers; the absolute values do not.",
    emphasize="Status",
    highlight_rows=lambda r: r["Status"] == "Fault injection",
)

## Four sentences to carry out of the room

The deck closes on these. They are worth reading now, before you have earned them, and again
at the end of notebook 09 when you have.

1. **Nothing downstream can recover a document the first stage never returned.**
2. **Index-time compute is paid once; query-time compute is paid forever.**
3. **An average is not a result until you have seen the slices underneath it.**
4. **Build the measurement before the improvement, every single time.**

---

**Next:** `01_retrieval_and_evaluation_foundations.ipynb` — the four-stage pipeline, the
recall budget, and the fault-isolation tree run against a real failing question.
